<a href="https://colab.research.google.com/github/shreyansh8h/hello-world/blob/master/May10_02_deliberative_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Deliberative Agents with LangChain and LLM Reasoning

## What you will learn

By the end, you should be able to:
- Explain deliberative agents as model-based planners.
- Build a planning tool.
- Use an LLM to reason about a generated plan.
- Replan when the world model changes.

This notebook uses LangChain with `HuggingFacePipeline` for real LLM reasoning. Set `HF_TOKEN` before running the LLM cells.

In [ ]:
# Install necessary libraries
%pip install -U transformers accelerate bitsandbytes langchain_community langchain_huggingface

In [ ]:
import os
import re
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.tools import tool
# from langchain_openai import ChatOpenAI # Commented out as we are using HuggingFace
from langchain_huggingface import HuggingFacePipeline
#from langchain_core.prompts import PromptTemplate
from google.colab import userdata

from transformers import pipeline

HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN: # Set HF_TOKEN as an environment variable if retrieved
    os.environ["HF_TOKEN"] = HF_TOKEN

def make_llm(temperature=0):
    if HF_TOKEN:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("Logged into Hugging Face.")
    else:
        print("Hugging Face token not found in Colab secrets. Proceeding without authentication.")

    # Initialize the HuggingFace pipeline for text generation with google/gemma-2b-it
    # 'token' is used for authentication with gated models on Hugging Face Hub
    # device_map='auto' helps distribute the model across available devices (CPU/GPU)
    hf_pipeline = pipeline(
        "text-generation",
        model="google/gemma-2b-it", # Corrected model name
        device_map="auto",
        token=os.getenv("HF_TOKEN"), # This will now correctly pick up the token from os.environ
        # Removed generation parameters from here to avoid passing to model's constructor
    )
    # Wrap the HuggingFace pipeline with LangChain's HuggingFacePipeline
    # Pass generation parameters via model_kwargs to HuggingFacePipeline
    llm = HuggingFacePipeline(
        pipeline=hf_pipeline,
        model_kwargs={
            "temperature": temperature,
            "do_sample": True, # Needed for temperature to have an effect
            "max_new_tokens": 500, # A reasonable default to prevent very short responses
        }
    )
    return llm


def print_box(title, text):
    print(f"\n{title}\n" + "-" * len(title))
    print(text)


print("LangChain setup ready.")
print("LLM provider: HuggingFace via HuggingFacePipeline") # Updated provider
print("Model:", "google/gemma-2b-it") # Explicitly state the model, corrected
print("HF_TOKEN configured:", bool(os.getenv("HF_TOKEN")))

LangChain setup ready.
LLM provider: HuggingFace via HuggingFacePipeline
Model: google/gemma-2b-it
HF_TOKEN configured: True


## 1. Define the World Model

In [ ]:
from collections import deque


ACTIONS = {
    "up": (0, -1),
    "down": (0, 1),
    "left": (-1, 0),
    "right": (1, 0),
}


class GridWorld:
    def __init__(self, width, height, blocked=None):
        self.width = width
        self.height = height
        self.blocked = set(blocked or [])

    def valid(self, position):
        x, y = position
        return 0 <= x < self.width and 0 <= y < self.height and position not in self.blocked

    def neighbors(self, position):
        for action, (dx, dy) in ACTIONS.items():
            next_position = (position[0] + dx, position[1] + dy)
            if self.valid(next_position):
                yield action, next_position

    def draw(self, start, goal, path=None):
        path = set(path or [])
        rows = []
        for y in range(self.height):
            row = []
            for x in range(self.width):
                pos = (x, y)
                if pos == start:
                    row.append("S")
                elif pos == goal:
                    row.append("G")
                elif pos in self.blocked:
                    row.append("#")
                elif pos in path:
                    row.append("*")
                else:
                    row.append(".")
            rows.append(" ".join(row))
        return "\n".join(rows)

## 2. Build a Planning Tool

In [ ]:
def breadth_first_plan(world, start, goal):
    frontier = deque([start])
    came_from = {start: None}
    action_from = {start: None}

    while frontier:
        current = frontier.popleft()
        if current == goal:
            break

        for action, next_position in world.neighbors(current):
            if next_position not in came_from:
                frontier.append(next_position)
                came_from[next_position] = current
                action_from[next_position] = action

    if goal not in came_from:
        return {"actions": [], "path": [], "status": "no_plan"}

    path = []
    actions = []
    current = goal
    while current != start:
        path.append(current)
        actions.append(action_from[current])
        current = came_from[current]
    path.append(start)

    return {
        "actions": list(reversed(actions)),
        "path": list(reversed(path)),
        "status": "planned",
    }


@tool
def route_planner(start: tuple, goal: tuple, blocked: list[tuple]) -> dict:
    """Plan a route on a 7x5 grid from start to goal while avoiding blocked cells."""
    world = GridWorld(width=7, height=5, blocked=blocked)
    return breadth_first_plan(world, start, goal)


print(route_planner.invoke({"start": (0, 0), "goal": (6, 4), "blocked": [(2, 0), (2, 1)]}))

{'actions': ['down', 'down', 'down', 'down', 'right', 'right', 'right', 'right', 'right', 'right'], 'path': [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 4), (3, 4), (4, 4), (5, 4), (6, 4)], 'status': 'planned'}


## 3. Create a Deliberative Agent Chain

The planning tool creates the plan. The LLM reasons about whether the plan is sensible.

In [ ]:
world = GridWorld(width=7, height=5, blocked={(2, 0), (2, 1), (2, 2), (4, 2), (4, 3)})
start = (0, 0)
goal = (6, 4)

plan_runnable = RunnableLambda(
    lambda inputs: {
        **inputs,
        "plan": breadth_first_plan(inputs["world"], inputs["start"], inputs["goal"]),
    }
)

llm = make_llm(temperature=0.1) # Changed temperature to a non-zero value to resolve ValueError

reasoning_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a planning critic. Explain whether the route plan is reasonable in two sentences."),
    ("human", "Start: {start}\nGoal: {goal}\nBlocked cells: {blocked}\nPlan: {plan}"),
])

plan_reasoning = (
    {
        "start": itemgetter("start"),
        "goal": itemgetter("goal"),
        "blocked": lambda x: sorted(x["world"].blocked),
        "plan": itemgetter("plan"),
    }
    | reasoning_prompt
    | llm
    | StrOutputParser()
)

deliberative_agent = RunnableParallel(
    planned_state=plan_runnable,
    reasoning=plan_runnable | plan_reasoning,
)

result = deliberative_agent.invoke({"world": world, "start": start, "goal": goal})
plan = result["planned_state"]["plan"]
print(world.draw(start, goal, plan["path"]))
print("\nActions:", plan["actions"])
print_box("LLM reasoning", result["reasoning"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged into Hugging Face.


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


S . # . . . .
* . # . . . .
* . # . # . .
* . . . # . .
* * * * * * G

Actions: ['down', 'down', 'down', 'down', 'right', 'right', 'right', 'right', 'right', 'right']

LLM reasoning
-------------
System: You are a planning critic. Explain whether the route plan is reasonable in two sentences.
Human: Start: (0, 0)
Goal: (6, 4)
Blocked cells: [(2, 0), (2, 1), (2, 2), (4, 2), (4, 3)]
Plan: {'actions': ['down', 'down', 'down', 'down', 'right', 'right', 'right', 'right', 'right', 'right'], 'path': [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 4), (3, 4), (4, 4), (5, 4), (6, 4)], 'status': 'planned'}

The route plan is not reasonable because it includes blocked cells and does not provide a clear path from the start to the goal.


## 4. Execute the Plan One Action at a Time

In [ ]:
def execute_plan(start, actions):
    position = start
    trace = [position]
    for action in actions:
        dx, dy = ACTIONS[action]
        position = (position[0] + dx, position[1] + dy)
        trace.append(position)
    return trace


trace = execute_plan(start, plan["actions"])
print("Final position:", trace[-1])
print(world.draw(start, goal, trace))

Final position: (6, 4)
S . # . . . .
* . # . . . .
* . # . # . .
* . . . # . .
* * * * * * G


## 5. Replan After a World Change

In [ ]:
changed_world = GridWorld(width=7, height=5, blocked=set(world.blocked) | {(0, 1)})
new_plan = breadth_first_plan(changed_world, start, goal)

replan_llm = make_llm(temperature=0.1) # Pass a non-zero temperature

replan_prompt = ChatPromptTemplate.from_messages([
    ("system", "Explain why replanning is needed in one or two sentences."),
    ("human", "Old plan: {old_plan}\nNew plan: {new_plan}\nNew blocked cells: {blocked}"),
])

replan_chain = replan_prompt | replan_llm | StrOutputParser()

print(changed_world.draw(start, goal, new_plan["path"]))
print("\nNew actions:", new_plan["actions"])
print_box(
    "LLM replanning explanation",
    replan_chain.invoke({
        "old_plan": plan["actions"],
        "new_plan": new_plan["actions"],
        "blocked": sorted(changed_world.blocked),
    }),
)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged into Hugging Face.


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


S * # . . . .
# * # . . . .
. * # . # . .
. * . . # . .
. * * * * * G

New actions: ['right', 'down', 'down', 'down', 'down', 'right', 'right', 'right', 'right', 'right']

LLM replanning explanation
--------------------------
System: Explain why replanning is needed in one or two sentences.
Human: Old plan: ['down', 'down', 'down', 'down', 'right', 'right', 'right', 'right', 'right', 'right']
New plan: ['right', 'down', 'down', 'down', 'down', 'right', 'right', 'right', 'right', 'right']
New blocked cells: [(0, 1), (2, 0), (2, 1), (2, 2), (4, 2), (4, 3)]

Replanning is needed because the new plan is not a valid path from the initial state to the final state. The new blocked cells prevent the robot from reaching the final position.


## Exercise

Change the goal and run the deliberative agent again.

Questions to answer:
- Did the plan become shorter or longer?
- Did the LLM explanation match the actual path?
- What could go wrong if the world model is stale?

In [ ]:
exercise_goal = (0, 4)
exercise_plan = breadth_first_plan(changed_world, start, exercise_goal)

print(changed_world.draw(start, exercise_goal, exercise_plan["path"]))
print("Actions:", exercise_plan["actions"])

S * # . . . .
# * # . . . .
. * # . # . .
. * . . # . .
G * . . . . .
Actions: ['right', 'down', 'down', 'down', 'down', 'left']


## Key Takeaway

A deliberative agent separates planning from execution. LangChain lets you expose planners as tools and use the LLM as a reasoning critic or explanation layer.

In [ ]:
exercise_goal = (3, 2)
exercise_plan = breadth_first_plan(changed_world, start, exercise_goal)

print(changed_world.draw(start, exercise_goal, exercise_plan["path"]))
print("Actions:", exercise_plan["actions"])

S * # . . . .
# * # . . . .
. * # G # . .
. * * * # . .
. . . . . . .
Actions: ['right', 'down', 'down', 'down', 'right', 'right', 'up']
